# 01 — Inspect candidate neurons and record competing hypotheses

**Rules for this notebook (see `PROTOCOL.md`):**

1. Only the **discovery split** is loaded here. Do not open `heldout_SEALED/` or any evaluation output.
2. For every neuron, write **both** H1 (≥2 independent features) and H2 (one abstraction), each with what it predicts **active** and **inactive**. "Neither" is a legitimate outcome; say so in `neither_note`.
3. Selection type (heterogeneity vs random baseline) is hidden on purpose. Don't look it up.
4. Freeze when done. Frozen records cannot be edited; a change becomes a new, reported version.

Prerequisite: `python scripts/run_discovery.py --config <config>`.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import HTML, display
from transformers import AutoTokenizer
from neurogen.config import load_config
from neurogen import pipeline as P, viz

CONFIG = ROOT / "configs" / "qwen3_4b.yaml"   # <- change to the config you ran
cfg = load_config(CONFIG, root=str(ROOT))
OUT = P.run_dir(cfg)
HYP_DIR = ROOT / "research" / "hypotheses"
print("run dir:", OUT)

In [ ]:
cands = P.load_candidates(OUT)
subset = json.loads((OUT / "discovery" / "phase3_subset.json").read_text())["cids"]
# Selection columns are dropped so the annotator stays blind to them.
table = pd.DataFrame(cands)[["cid", "layer", "neuron", "polarity", "active_threshold"]]
table["phase3"] = table.cid.isin(subset)
table.sort_values("cid")

In [ ]:
tok = AutoTokenizer.from_pretrained(cfg.model.name, revision=cfg.model.revision)
disc, _held = P.load_corpora(cfg, tok)          # held-out split is NOT used in this notebook
del _held
traces = np.load(OUT / "discovery" / "traces.npz")
genres = P.genre_array(disc)
print(f"discovery: {disc.n_tokens} tokens, {len(disc.docs)} docs")

## Pick a neuron
Work through the Phase 3 subset first (those will be tested), then the rest if time allows.

In [ ]:
CID = sorted(subset)[0]          # <- change
cand = next(c for c in cands if c["cid"] == CID)
cards = P.load_cards(OUT, CID)
print(CID, "| layer", cand["layer"], "neuron", cand["neuron"], "polarity", cand["polarity"])
display(HTML(viz.cards_html(cards)))

### Where does it fire? (discovery split, all tokens)
Box = full distribution per genre; dots = tokens above the pre-registered active threshold; % = fraction active.

In [ ]:
viz.plot_genre_distribution(traces[CID], genres, cand["polarity"], cand["active_threshold"])
plt.tight_layout(); plt.show()

### Optional: input geometry at the top-activating tokens
Spherical 2-means on the MLP inputs `x` at tokens above the k-th largest activation. Well-separated clusters are *compatible* with H1, one cluster with H2; **neither is decisive**. Use it to sharpen hypotheses, not to judge them.

In [ ]:
from neurogen.analysis import two_cluster_separation
geo = np.load(OUT / "discovery" / "geometry.npz")
if f"{CID}__x" in geo:
    sep = two_cluster_separation(geo[f"{CID}__x"], seed=cfg.seed)
    gids = geo[f"{CID}__gid"]
    rows = []
    for g, lab in zip(gids, sep["labels"]):
        d, pos = disc.locate(int(g))
        rows.append(dict(cluster=lab, genre=disc.docs[d].genre, token=tok.decode([disc.ids[d][pos]])))
    print(f"silhouette {sep['silhouette']:.2f}, sizes {sep['sizes']}")
    display(pd.DataFrame(rows).groupby(["cluster", "genre"]).size().unstack(fill_value=0))
else:
    print("no geometry vectors saved for", CID)

## Record hypotheses
Example ids refer to the `#j` numbers in the cards above. Be concrete: "predicts active" / "predicts inactive" should let someone else write test sentences without asking you.

*The text below is a placeholder showing the format. It is **not** a finding.*

In [ ]:
from neurogen.hypotheses import Feature, HypothesisRecord, save_record
rec = HypothesisRecord(
    cid=CID, layer=cand["layer"], neuron=cand["neuron"], polarity=cand["polarity"],
    researcher="YOUR INITIALS",
    h1_features=[
        Feature("feature A", "describe A", example_ids=[0, 2]),
        Feature("feature B", "describe B", example_ids=[1, 3]),
    ],
    h1_predicts_active="instances of A, or of B, even when they share nothing else",
    h1_predicts_inactive="contexts with neither A nor B, incl. ones that share C's abstract property",
    h2_abstraction="abstraction C",
    h2_description="describe C without listing A and B",
    h2_example_ids=[0, 1, 2, 3],
    h2_predicts_active="any instance of C, including ones that look like neither A nor B",
    h2_predicts_inactive="surface instances of A or B that lack C",
    unexplained_example_ids=[],
    neither_note="",
    prior={"h1": 0.4, "h2": 0.4, "neither": 0.2},
    discovery_cards_sha=cand["discovery_cards_sha"],
)
print("problems:", rec.problems() or "none")
# save_record(rec, HYP_DIR)      # <- uncomment to save

## Freeze
Run once all Phase 3 neurons have records. After this, run `python scripts/prepare_evaluation.py --stage tests`.

In [ ]:
from neurogen.hypotheses import freeze
# freeze(HYP_DIR, cids=subset)   # <- uncomment to freeze